# NAMAA V4 clean environment. One flat top-level code cell. Matched torch and torchaudio 2.6.0 CUDA12.6 wheels, fresh Python process, check actual WAV output.

In [ ]:
import os,sys,subprocess
from pathlib import Path
venv=Path("/kaggle/working/namaa_venv_v4")
env=os.environ.copy()
env["PIP_NO_CACHE_DIR"]="1"
env["HF_HOME"]="/kaggle/working/namaa_hf_cache"
print("PHASE_CREATE_VENV",flush=True)
subprocess.run([sys.executable,"-m","venv",str(venv)],check=True)
py=str(venv/"bin/python")
print("PHASE_PIN_PYTORCH_2_6_CU126",flush=True)
subprocess.run([py,"-m","pip","install","-q","torch==2.6.0","torchaudio==2.6.0","--index-url","https://download.pytorch.org/whl/cu126"],check=True,env=env)
print("PHASE_INSTALL_CHATTERBOX_0_1_7",flush=True)
subprocess.run([py,"-m","pip","install","-q","chatterbox-tts==0.1.7","huggingface_hub"],check=True,env=env)
print("PHASE_PREFLIGHT",flush=True)
subprocess.run([py,"-c","import torch,torchaudio; from chatterbox import mtl_tts; print('PRECHECK_OK',torch.__version__,torchaudio.__version__,torch.cuda.is_available(),flush=True); assert torch.cuda.is_available()"],check=True,env=env)
inference=Path("/kaggle/working/namaa_infer_v4.py")
inference.write_text("import os,time,sys\nfrom pathlib import Path\nimport torch,torchaudio\nfrom huggingface_hub import snapshot_download\nfrom safetensors.torch import load_file\nfrom chatterbox import mtl_tts\nprint(\"TORCH\",torch.__version__,\"TORCHAUDIO\",torchaudio.__version__,\"GPU\",torch.cuda.is_available(),flush=True)\nassert torch.__version__.split(\"+\")[0]==\"2.6.0\"\nassert torchaudio.__version__.split(\"+\")[0]==\"2.6.0\"\nassert torch.cuda.is_available(), \"GPU required\"\nprint(\"BASE_LOADING\",flush=True)\nstart=time.monotonic()\nengine=mtl_tts.ChatterboxMultilingualTTS.from_pretrained(device=\"cuda\")\nprint(\"BASE_READY\",round(time.monotonic()-start,2),flush=True)\nfolder=snapshot_download(repo_id=\"NAMAA-Space/NAMAA-Egyptian-TTS\",allow_patterns=[\"t3_mtl23ls_v2.safetensors\"])\nengine.t3.load_state_dict(load_file(str(Path(folder)/\"t3_mtl23ls_v2.safetensors\"),device=\"cpu\"),strict=True)\nengine.t3.to(\"cuda\").eval()\nprint(\"NAMAA_READY\",flush=True)\nt=time.monotonic()\nwith torch.inference_mode():\n    audio=engine.generate(\"يا أهلا بيكم، دي تجربة صوت مصري حقيقي، وهنحكي قصة مانسا موسى.\",language_id=\"ar\")\nout=Path(\"/kaggle/working/namaa_test.wav\")\ntorchaudio.save(str(out),audio.detach().cpu(),engine.sr)\nprint(\"SUCCESS\",str(out),\"BYTES\",out.stat().st_size,\"AUDIO_SECONDS\",round(audio.shape[-1]/engine.sr,2),\"INFER_SECONDS\",round(time.monotonic()-t,2),\"TOTAL_SECONDS\",round(time.monotonic()-start,2),flush=True)\n",encoding="utf-8")
print("PHASE_NAMAA_INFERENCE",flush=True)
subprocess.run([py,"-u",str(inference)],check=True,env=env)
